In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.functions import (col, date_trunc, md5,
                                   current_timestamp, coalesce)
from datetime import datetime

# Widget with default value
dbutils.widgets.text("table_name", "dim_products")
table_name = dbutils.widgets.get("table_name")

# Configuration & Paths
silver_base = "abfss://processed@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/silver/"
gold_base = "abfss://curated@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/gold/"
target_table = table_name
gold_path = f"{gold_base}{target_table}"

# Silver Sources
df_prod = spark.read.format("delta").load(f"{silver_base}olist_products_dataset")
df_translation = spark.read.format("delta").load(f"{silver_base}product_category_name_translation")

# Transformation logic
df_dim_base = (df_prod.alias("p")
               .join(df_translation.alias("t"), "product_category_name", "left")
               .select(
                  F.md5(F.col("p.product_id")).cast("string").alias("product_key"),
                  "p.product_id",
                  F.coalesce(F.col("p.product_category_name"), F.lit("outros")).alias("product_category_name"),
                  F.coalesce(F.col("t.product_category_name_english"), F.lit("others")).alias("product_category_name_english"),
                  "p.product_weight_g",
                  "p.product_length_cm",
                  "p.product_height_cm",
                  "p.product_width_cm",
                  "p.product_volume_cm3",
                  F.date_trunc("second", F.current_timestamp()).alias("gold_load_at")
               ))

# Unknown record
now = datetime.now().replace(microsecond=0)
unknown_data = [(-1, "UNKNOWN", "outros", "others", 0, 0, 0, 0, 0, now)]
target_schema = df_dim_base.schema
df_unknown = spark.createDataFrame(unknown_data, schema=target_schema)
df_final = df_dim_base.unionByName(df_unknown)

# Incremental Merge (Upsert)
print(f"--> Starting Gold Load: {target_table}")

if DeltaTable.isDeltaTable(spark, gold_path):
    dt_gold = DeltaTable.forPath(spark, gold_path)

    (dt_gold.alias("target")
     .merge(df_final.alias("source"), "target.product_id = source.product_id")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
    
    print(f"--> [Merge] {target_table} updated.")
else:
    #First time load
    df_final.write.format("delta").mode("overwrite").save(gold_path)
    print(f"--> [INIT] {target_table} created.")

# Audit & Exit
dt_final = DeltaTable.forPath(spark, gold_path)

last_operation = dt_final.history(1).select("operationMetrics").collect()[0][0]
rows_inserted = int(last_operation.get("numTargetRowsInserted", 0))
rows_updated = int(last_operation.get("numTargetRowsUpdated", 0))
total_rows = last_operation.get("numOutputRows", "Check History")

print("-" * 50)
print(f"--> Table: {table_name} | Status: Success")
print(f"--> Rows Processed in last run: {rows_inserted + rows_updated}")
print("-" * 50)

dbutils.notebook.exit(f"Success | Inserted: {rows_inserted} | Updated: {rows_updated}")